In [ ]:
import numpy as np
import h5py

from netfinal import Net

import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
h5_path = "../../data/real_0.h5"
!file $h5_path

In [ ]:
h5_export_path = "inference/bem_gt_pred_pairings__net_real_0.h5"
#!rm -f $h5_export_path

In [ ]:
weights_path = "checkpoints/2026-08-20_20-58-59/epoch_7.pth"

In [ ]:
model = Net(generate_vel=False, generate_depthmap=True, train_unet=True, train_velpred=False, use_convtrans=True, skip_decoder=False).to(device)

In [ ]:
state_dict = torch.load(weights_path, map_location=device, weights_only=True)["model_state_dict"]
model.load_state_dict(state_dict)
model.eval()

In [ ]:
class H5ReadWrapper:
    def __init__(self, path, shuffle=True):
        self.path = path
        self.shuffle = shuffle

        with h5py.File(path, "r") as f:
            self.len = len(f.keys())

    def __len__(self):
        return self.len

    def __iter__(self):
        with h5py.File(self.path, "r") as f:
            traj_ids = list(f.keys())
            if self.shuffle:
                random.shuffle(traj_ids)
    
            for traj_id in traj_ids:
                traj = f[traj_id]
                yield traj_id, (traj["trajlength"][()] - 1, traj["depths"][1:], traj["evs"][:])

In [ ]:
enumerate_frames = lambda traj: ((depth, ev) for depth, ev in zip(*traj[1:]))
prep_image = lambda im: torch.from_numpy(im).to(device).unsqueeze(0).unsqueeze(0)
unprep_image = lambda im: im.squeeze().squeeze().cpu().numpy()

In [ ]:
def calculate_loss(gt, pred, eps=1e-8):
    mask = gt > 0

    valid_gt = gt[mask]
    valid_pred = pred[mask]

    pixel_loss = ((valid_gt - valid_pred) ** 2) / (valid_gt + eps)
    return pixel_loss.mean().cpu(), pixel_loss.median().cpu()

In [ ]:
def calculate_stats(gt, pred, eps=1e-8):
    mask = gt > 0

    valid_gt = pred[mask]
    valid_pred = gt[mask]

    valid_gt = valid_gt.clamp(min=eps)
    valid_pred = valid_pred.clamp(min=eps)
    
    mae = (valid_gt - valid_pred).abs().mean()
    
    rmse = ((valid_gt - valid_pred) ** 2).mean().sqrt()
    
    abs_rel = ((valid_gt - valid_pred).abs() / valid_gt).mean()
    
    thresh = torch.maximum((valid_pred / valid_gt), (valid_gt / valid_pred))
    delta1 = (thresh < 1.25).float().mean()

    return mae.cpu(), rmse.cpu(), abs_rel.cpu(), delta1.cpu()

In [ ]:
with torch.no_grad():
    prep_image = lambda im: torch.from_numpy(im).to(device).unsqueeze(0).unsqueeze(0)
    unprep_image = lambda im: im.squeeze().squeeze().cpu().numpy()

    for traj_name, trajectory in H5ReadWrapper(h5_path, False):
        h = (
            torch.zeros(1, 128, 10, 10, device=device),
            torch.zeros(1, 128, 10, 10, device=device),
            torch.zeros(1, 128, 10, 10, device=device),
            torch.zeros(1, 128, 10, 10, device=device),
            torch.zeros(1, 128, 10, 10, device=device),
            torch.zeros(1, 128, 10, 10, device=device),
            torch.zeros(1, 128, 10, 10, device=device),
            torch.zeros(1, 128, 10, 10, device=device)
        )
        
        visualization_history = []

        evs = []
        gts = []
        preds = []
        stats = []
    
        freq = (trajectory[0] // 5) + 1
        
        for i, (depth, ev) in enumerate(enumerate_frames(trajectory)):
            depth_prep = prep_image(depth)
            ev_prep = prep_image(ev)

            depth_pred, *h = model(depth_prep, *h)

            depth_pred *= (depth_prep > 0).float()
            depth_pred = F.relu(depth_pred)

            depth_pred_post = unprep_image(depth_pred)

            loss = calculate_loss(depth_prep, depth_pred)
            sts = calculate_stats(depth_prep, depth_pred)

            evs.append(ev)
            gts.append(depth_post)
            preds.append(depth)
            stats.append(np.array(loss + sts))
            
            if i % freq == 0:    
                visualization_history.append({
                    "bem": (ev != 0).astype(int),
                    "pred": depth_pred_post,
                    "truth": depth,
                    "loss": loss,
                    "stats": sts,
                    "iter": i + 1
                })
    
        num_rows = len(visualization_history)
    
        fig, axes = plt.subplots(num_rows, 3, figsize=(12, 4 * num_rows))
        
        if num_rows == 1:
            axes = [axes]
        
        for i, data in enumerate(visualization_history):
            ax_bem = axes[i][0]
            ax_pred = axes[i][2]
            ax_true = axes[i][1]
    
            im0 = ax_bem.imshow(data["bem"], cmap="gray")
            ax_bem.set_title(f"Iter {data["iter"]}")
            ax_bem.axis("off")
            
            im1 = ax_pred.imshow(data["pred"], cmap="gray")
            ax_pred.set_title(f"Loss {data["loss"][0]:.2f}")
            ax_pred.axis("off")
            
            im2 = ax_true.imshow(data["truth"], cmap="gray")
            ax_true.set_title(f"delta1 {100 * data["stats"][3]:.2f}")
            ax_true.axis("off")

        fig.suptitle(f"Trajectory: {traj_name}\n")
        plt.tight_layout()
        plt.show()

        with h5py.File(h5_export_path, 'a') as f:
            if traj_name in f:
                del f[traj_name]

            group = f.create_group(traj_name)

            group.create_dataset("num_labels", data=trajectory[0])
            stats_d = group.create_dataset("stats", data=np.stack(stats, axis=0))
            evs_d = group.create_dataset("evs", data=np.stack(evs, axis=0), compression="gzip")
            gts_d = group.create_dataset("gts", data=np.stack(gts, axis=0), compression="gzip")
            preds_d = group.create_dataset("preds", data=np.stack(preds, axis=0), compression="gzip")

            stats_d.attrs["encoding"] = "meanloss,medianloss,mae,rmse,absrel,delta1"
            evs_d.attrs["scale"] = 0.2
            gts_d.attrs["units"] = "m"
            gts_d.attrs["scale"] = 1/30
            preds_d.attrs["units"] = "m"
            preds_d.attrs["scale"] = 1/30